# 📡 Project 11.1 — Sensor Log Binary Search
**DSA for Mechatronics · Week 11 — Searching Algorithms**

> **Run all cells top to bottom. Submit the .ipynb on Blackboard.**
> The final cell prints your personalised report — be ready to explain every step.


In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║            ENTER YOUR STUDENT ID BELOW              ║
# ╚══════════════════════════════════════════════════════╝

STUDENT_ID = "12345678"   # ← replace with your real student ID

import random, hashlib, bisect, time
_seed = int(hashlib.md5(STUDENT_ID.encode()).hexdigest(), 16) % (2**31)
random.seed(_seed)
print(f"Student ID  : {STUDENT_ID}")
print(f"Dataset seed: {_seed}")
print("✅ Your personal dataset is ready.")


## The Scenario
A data logger records sensor readings sorted by timestamp (integer ms values).
We search this log using four methods, measure the step counts, and compare:

1. **Linear search** — scan from left, O(n)
2. **Binary search** — halve search space each step, O(log n)
3. **Boundary search** — find first and last occurrence of a repeated value
4. **Search insert position** — find where a new reading belongs to keep log sorted
5. **Python `bisect` module** — production-quality verification


## Step 1 — Build the sorted sensor log

In [ ]:
# ── Personalised parameters ──────────────────────────────────────
N_LOG      = random.randint(40, 70)
VAL_RANGE  = random.randint(100, 200)
N_TARGETS  = random.randint(5, 8)

# Build sorted log (allow duplicates — some values repeated 2–3×)
raw = []
for _ in range(N_LOG):
    v = random.randint(1, VAL_RANGE)
    # occasionally duplicate
    raw.append(v)
    if random.random() < 0.15:
        raw.append(v)
raw = sorted(raw[:N_LOG])   # trim and sort

N_LOG = len(raw)
LOG = raw

# Pick search targets: mix of present and absent values
present = random.sample(list(set(LOG)), min(N_TARGETS - 2, len(set(LOG))))
absent  = []
for _ in range(2):
    v = random.randint(1, VAL_RANGE)
    while v in set(LOG):
        v = random.randint(1, VAL_RANGE)
    absent.append(v)
TARGETS = present + absent
random.shuffle(TARGETS)

print(f"Sorted sensor log:")
print(f"  Entries  : {N_LOG}")
print(f"  Range    : 1 – {VAL_RANGE}")
print(f"  Unique   : {len(set(LOG))}")
print(f"  Log (first 20): {LOG[:20]} ...")
print()
print(f"  Search targets: {TARGETS}")
print(f"  ({len(present)} present, {len(absent)} absent)")


## Step 2 — Linear search vs Binary search (with step counting)

In [ ]:
def linear_search(arr, target):
    """
    Linear search: scan left to right, compare each element.
    O(n) worst case — must examine every element if target absent.
    Returns (index, steps).
    """
    for i, v in enumerate(arr):
        if v == target:
            return i, i + 1          # found at step i+1
    return -1, len(arr)              # exhausted entire array

def binary_search(arr, target):
    """
    Binary search: maintain [lo, hi] window.
    Each step compare arr[mid] to target:
      - Equal  → found
      - Too small → search right half  (lo = mid + 1)
      - Too large → search left half   (hi = mid - 1)
    O(log n) — window halves every step.
    Returns (index, steps).
    """
    lo, hi = 0, len(arr) - 1
    steps = 0
    while lo <= hi:
        steps += 1
        mid = lo + (hi - lo) // 2        # avoids overflow in other languages
        if arr[mid] == target:
            return mid, steps
        elif arr[mid] < target:
            lo = mid + 1
        else:
            hi = mid - 1
    return -1, steps

print(f"Linear vs Binary search (n={N_LOG}, log₂n ≈ {N_LOG.bit_length()}):")
print()
print(f"  {'Target':>8}  {'Lin idx':>8}  {'Lin steps':>10}  {'Bin idx':>8}  {'Bin steps':>10}  {'Speedup':>8}")
print(f"  {'─'*8}  {'─'*8}  {'─'*10}  {'─'*8}  {'─'*10}  {'─'*8}")

lin_steps_total = bin_steps_total = 0
search_results = {}
for t in TARGETS:
    li, ls = linear_search(LOG, t)
    bi, bs = binary_search(LOG, t)
    lin_steps_total += ls
    bin_steps_total += bs
    speedup = round(ls / bs, 1) if bs > 0 else "—"
    status = "✅" if (li == -1) == (bi == -1) and (bi == -1 or LOG[bi] == t) else "❌"
    search_results[t] = (li, ls, bi, bs)
    print(f"  {t:>8}  {li:>8}  {ls:>10}  {bi:>8}  {bs:>10}  {str(speedup):>8}  {status}")

print()
print(f"  Total steps — Linear: {lin_steps_total}   Binary: {bin_steps_total}")
avg_speedup = round(lin_steps_total / bin_steps_total, 1) if bin_steps_total > 0 else 0
print(f"  Average speedup      : {avg_speedup}×")
print(f"  Theoretical max steps: linear={N_LOG}  binary={N_LOG.bit_length()}")


## Step 3 — Boundary search: first and last occurrence

In [ ]:
def find_first(arr, target):
    """
    Find the FIRST index where arr[index] == target.
    Modified binary search: when found, don't stop — move hi = mid - 1
    to keep searching left half for an earlier occurrence.
    O(log n).
    """
    lo, hi = 0, len(arr) - 1
    result = -1
    while lo <= hi:
        mid = lo + (hi - lo) // 2
        if arr[mid] == target:
            result = mid
            hi = mid - 1      # ← keep searching LEFT
        elif arr[mid] < target:
            lo = mid + 1
        else:
            hi = mid - 1
    return result

def find_last(arr, target):
    """
    Find the LAST index where arr[index] == target.
    When found, move lo = mid + 1 to keep searching right half.
    O(log n).
    """
    lo, hi = 0, len(arr) - 1
    result = -1
    while lo <= hi:
        mid = lo + (hi - lo) // 2
        if arr[mid] == target:
            result = mid
            lo = mid + 1      # ← keep searching RIGHT
        elif arr[mid] < target:
            lo = mid + 1
        else:
            hi = mid - 1
    return result

# Find values that appear more than once
duplicates = [v for v in set(LOG) if LOG.count(v) > 1]
boundary_targets = duplicates[:4] if len(duplicates) >= 4 else duplicates
# add one unique value for contrast
unique_vals = [v for v in set(LOG) if LOG.count(v) == 1]
if unique_vals:
    boundary_targets.append(random.choice(unique_vals))

print(f"Boundary search — first and last occurrence:")
print()
print(f"  {'Value':>7}  {'Count':>6}  {'First':>7}  {'Last':>7}  {'Range':>12}  bisect verify")
print(f"  {'─'*7}  {'─'*6}  {'─'*7}  {'─'*7}  {'─'*12}  {'─'*14}")
for v in boundary_targets:
    f = find_first(LOG, v)
    l = find_last(LOG, v)
    cnt = LOG.count(v)
    span = l - f + 1 if f != -1 else 0
    b_lo = bisect.bisect_left(LOG, v)
    b_hi = bisect.bisect_right(LOG, v) - 1
    ok = (f == b_lo and l == b_hi) if f != -1 else True
    print(f"  {v:>7}  {cnt:>6}  {f:>7}  {l:>7}  [{f:3d}–{l:3d}] len={span}  {'✅' if ok else '❌'}")


## Step 4 — Search insert position + timing comparison

In [ ]:
def search_insert(arr, target):
    """
    Find index where target should be inserted to keep arr sorted.
    Equivalent to lower_bound / bisect_left.

    When lo > hi (not found), lo is the insertion point:
    - All elements to left are < target
    - All elements to right are >= target
    O(log n).
    """
    lo, hi = 0, len(arr) - 1
    while lo <= hi:
        mid = lo + (hi - lo) // 2
        if arr[mid] == target:
            return mid          # exact match: insert here (or replace)
        elif arr[mid] < target:
            lo = mid + 1
        else:
            hi = mid - 1
    return lo                   # not found: lo is insertion point

# Generate insert targets (some present, some absent)
insert_targets = [random.randint(1, VAL_RANGE) for _ in range(8)]

print(f"Search insert position (n={N_LOG}):")
print()
print(f"  {'Target':>8}  {'Insert pos':>11}  {'bisect_left':>12}  {'Match':>6}  Interpretation")
print(f"  {'─'*8}  {'─'*11}  {'─'*12}  {'─'*6}  {'─'*30}")
for t in insert_targets:
    pos  = search_insert(LOG, t)
    bpos = bisect.bisect_left(LOG, t)
    ok   = pos == bpos
    if t in set(LOG):
        interp = f"found at {pos}"
    else:
        left  = LOG[pos-1] if pos > 0 else "—"
        right = LOG[pos] if pos < len(LOG) else "—"
        interp = f"{left} < {t} ≤ {right}"
    print(f"  {t:>8}  {pos:>11}  {bpos:>12}  {'✅' if ok else '❌':>6}  {interp}")

# Timing: linear vs binary on full log
N_TIMING = 5000
timing_targets = [random.randint(1, VAL_RANGE) for _ in range(N_TIMING)]
t0 = time.perf_counter()
for t in timing_targets: linear_search(LOG, t)
lin_ms = (time.perf_counter() - t0) * 1000
t0 = time.perf_counter()
for t in timing_targets: binary_search(LOG, t)
bin_ms = (time.perf_counter() - t0) * 1000

print()
print(f"Timing ({N_TIMING} random searches on n={N_LOG} log):")
print(f"  Linear search  : {lin_ms:.2f} ms")
print(f"  Binary search  : {bin_ms:.2f} ms")
print(f"  Speedup        : {lin_ms/bin_ms:.1f}×")


## 📋 Final Report

In [ ]:
W = 58
print("╔" + "═"*W + "╗")
print("║" + " SENSOR LOG BINARY SEARCH — REPORT".center(W) + "║")
print("╠" + "═"*W + "╣")
print(f"║  {'Student ID':<28}: {STUDENT_ID:<26}║")
print(f"║  {'Dataset seed':<28}: {_seed:<26}║")
print("╠" + "═"*W + "╣")
print("║  DATASET" + " "*(W-9) + "║")
print(f"║  {'Log entries (n)':<28}: {N_LOG:<26}║")
print(f"║  {'Value range':<28}: 1 – {VAL_RANGE:<22}║")
print(f"║  {'Unique values':<28}: {len(set(LOG)):<26}║")
print("╠" + "═"*W + "╣")
print("║  LINEAR vs BINARY SEARCH" + " "*(W-25) + "║")
print(f"║  {'Targets searched':<28}: {len(TARGETS):<26}║")
print(f"║  {'Total linear steps':<28}: {lin_steps_total:<26}║")
print(f"║  {'Total binary steps':<28}: {bin_steps_total:<26}║")
print(f"║  {'Average speedup':<28}: {avg_speedup}×{'':<24}║")
print(f"║  {'Theoretical max (binary)':<28}: {N_LOG.bit_length()} steps{'':<19}║")
print("╠" + "═"*W + "╣")
print(f"║  TIMING ({N_TIMING} SEARCHES)" + " "*(W-22) + "║")
print(f"║  {'Linear':<28}: {lin_ms:.2f} ms{'':<20}║")
print(f"║  {'Binary':<28}: {bin_ms:.2f} ms{'':<20}║")
print(f"║  {'Speedup':<28}: {lin_ms/bin_ms:.1f}×{'':<24}║")
print("╚" + "═"*W + "╝")


---
## 📝 Reflection — answer before submitting

Double-click this cell to edit. Write 2–4 sentences for each question.

---

**Q1 — What does your output show?**
Describe the key results from your final report. What does it tell you about searching in this context?

*Your answer here:*

---

**Q2 — Why is binary search O(log n) and not O(n)?**
Explain in your own words why halving the search space each step leads to logarithmic complexity,
and give one example from your results that illustrates this.

*Your answer here:*

---

**Q3 — What would change if your student ID changed?**
Which specific numbers or results in the final report would be different, and why?

*Your answer here:*
